# 56장 · 도구·구조화 응답·실행 정책 연결하기

검색 도구가 대출 문서를 두 장 달라는 요청을 받았습니다. 그런데 화면이 정수 2 대신 문자열 ‘2’를 보냈습니다. 함수가 받기 전의 검사와 답변을 내보내기 전의 검사를 따로 만들어 봅니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain 1.4.6, langchain-core 1.6.12, langsmith 0.14.11입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 검색을 시작하기 전에 막을 입력

수업의 search_documents는 k에 양의 정수를 받습니다. 기존 SearchArgs는 StrictInt로 이 약속을 검사합니다. 아래에서는 같은 조건을 Python에서 풀어 써서 어떤 값 때문에 멈추는지 봅니다.


### 셀 01 · 문자열로 온 검색 개수

문자열을 검사 함수에 보내고 어떤 오류 메시지가 나오는지 보세요.

In [1]:
def check_k(value):
    if type(value) is not int or value < 1:
        raise ValueError("k에 양의 정수를 넣으세요.")
    return value

try:
    check_k("2")
except ValueError as error:
    print(error)

k에 양의 정수를 넣으세요.


화면에서 숫자처럼 보여도 문자열에는 따옴표가 붙습니다. 요청을 거절한 시점에는 검색 호출이 발생하지 않았습니다. True도 정수로 잘못 받지 않도록 타입을 확인했습니다.

### 과제 02 · 유효한 검색 한도 보내기

요청을 고쳐 문서 두 장까지 검색할 수 있게 하세요.

In [2]:
# check_k에 값을 전달하고 결과를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(check_k(2))</code></pre>
<p>실행 결과</p><pre><code>2</code></pre>
<p>정수 2가 그대로 돌아옵니다. 두 장을 요청할 수 있다는 뜻이며 결과가 반드시 두 장이라는 보장은 없습니다.</p>
</details>

## 문서 이름이 실제 검색 결과에 있을까요?

형식에 맞는 답변도 존재하지 않는 문서 번호를 인용할 수 있습니다. 검색이 끝나면 실제 반환한 자료 범위 안에서만 cited_ids를 허용합니다. 원문이 문장을 뒷받침하는지는 별도로 읽어야 합니다.


### 셀 03 · 없는 문서 번호 확인

검색된 ID 하나와 초안이 인용한 ID 하나를 대조하세요.

In [3]:
returned_ids = {"library-1"}
draft_ids = {"library-9"}
print(sorted(draft_ids - returned_ids))

['library-9']


library-9는 현재 검색에서 받은 적이 없습니다. 이 경우에는 자료가 없었다고 기록하면 안 됩니다. 검색은 성공했고 초안의 인용 검사에서 막혔습니다.

### 과제 04 · 근거가 있는 인용 통과시키기

초안이 반환된 문서만 인용하는지 집합으로 검사하세요.

집합에서 <=는 왼쪽의 모든 원소가 오른쪽에도 있는지를 묻습니다. library-1이 반환 ID에 들어 있으면 인용 검사는 통과합니다. 대출 일수까지 맞는지 확인하려면 library-1의 본문과 답의 숫자를 대조해야 합니다.

In [4]:
cited = {'library-1'}
# 모든 ID가 returned_ids 안에 있는지 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>cited = {&#x27;library-1&#x27;}
print(cited &lt;= returned_ids)</code></pre>
<p>실행 결과</p><pre><code>True</code></pre>
<p>True가 나옵니다. 이제 해당 원문의 ‘14일’과 답변의 일수를 사람이 대조할 차례입니다.</p>
</details>

## 한도를 쓴 다음 요청에서 멈추기

실행 중에 쌓이는 메시지와 횟수가 상태입니다. 도구를 몇 번까지 허용할지는 호출자가 미리 정하는 설정입니다. 실제 LangChain 미들웨어에 한도를 주고, 호출 흔적을 남기는 함수를 등록해 두 번째 요청이 함수까지 도착하는지 봅니다.


### 셀 05 · 제공 셀 · 반복 요청 대본 준비

모델 대신 같은 요청을 두 번 내보낼 대본을 준비합니다.

In [5]:
from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage
from langsmith import tracing_context

class ClassroomModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, **kwargs):
        return self  # 사용할 도구 요청은 아래 대본에 고정했습니다.

def ask_tool(name, args, call_id):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

응답할 메시지만 준비했습니다. 아직 검색 함수는 한 번도 실행하지 않았습니다.

### 셀 06 · 실제 미들웨어에서 두 번째 실행 막기

마지막 호출 흔적에 무엇이 몇 번 남을지 예상하세요.

In [6]:
from langchain.agents.middleware import ToolCallLimitMiddleware

executed = []
def search_documents(question: str) -> str:
    """가상 대출 규정을 한 건 조회한다."""
    executed.append(question)
    return "도서 대출 기간은 14일입니다."

model = ClassroomModel(responses=[
    ask_tool("search_documents", {"question": "대출"}, "s1"),
    ask_tool("search_documents", {"question": "다시 검색"}, "s2"),
    AIMessage(content="답변을 마칩니다."),
])
limit = ToolCallLimitMiddleware(run_limit=1, exit_behavior="continue")
agent = create_agent(model, tools=[search_documents], middleware=[limit])
with tracing_context(enabled=False):
    state = agent.invoke({"messages": [("user", "대출 규정을 확인해 주세요.")]})
print(executed)

['대출']


['대출'] 한 건만 남습니다. 두 번째 요청도 대본에 있었지만 미들웨어가 실행 상한을 먼저 검사했습니다. 검색 함수 안에 append를 둔 덕분에 모델이 요청한 횟수와 실제 실행한 횟수를 구분할 수 있습니다.

## 돌아보기

입력 오류, 정상적으로 찾은 자료가 0건인 경우, 인용 오류를 각각 어느 단계에서 알아낼 수 있는지 말해 보세요. 실제 예제 policy_agent.py는 도구 실행 전후의 검사와 호출 상한을 미들웨어에 둡니다.